# Supervised Machine Learning Assessment  
## Pharma: Drug Shelf-Life / Demand Prediction

This corrected notebook covers the complete assessment workflow:

1. Load and understand the dataset  
2. Handle missing values and duplicates  
3. Perform normal EDA  
4. Detect and cap outliers  
5. Perform basic feature engineering  
6. Encode categorical variables using **LabelEncoder**  
7. Build Linear Regression  
8. Build Random Forest Regressor  
9. Compare with Gradient Boosting Regressor  
10. Tune model using GridSearchCV  
11. Evaluate using RMSE, MAE, R², and cross-validation  

> Run the notebook from top to bottom.

## 0. Import libraries

In [ ]:
import os
import zipfile
import warnings
from pathlib import Path
from io import BytesIO

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from IPython.display import display

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

## 1. Load dataset

The code first checks for a local CSV file.  
If the file is not available locally, it downloads the ZIP file from the GitHub dataset link and loads the first CSV or Excel file inside it.

In [ ]:
DATA_URL = "https://github.com/bhaumik2025/DS_Data/raw/main/Datasets/Supervised%20ML/Pharmaceutical%20Supply%20Chain%20Optimization.zip"

possible_local_files = [
    "Pharmaceutical Supply Chain Optimization.csv",
    "pharmaceutical_supply_chain_optimization.csv"
]

df = None

# 1) Try loading local CSV first
for file_name in possible_local_files:
    if os.path.exists(file_name):
        df = pd.read_csv(file_name)
        print(f"Loaded local file: {file_name}")
        break

# 2) If local CSV is not found, download ZIP from GitHub
if df is None:
    print("Local CSV not found. Downloading dataset from GitHub...")
    response = requests.get(DATA_URL)
    response.raise_for_status()

    zip_file = zipfile.ZipFile(BytesIO(response.content))
    print("Files inside ZIP:")
    print(zip_file.namelist())

    data_files = [
        file for file in zip_file.namelist()
        if file.lower().endswith((".csv", ".xlsx", ".xls"))
    ]

    if len(data_files) == 0:
        raise FileNotFoundError("No CSV or Excel file found inside the ZIP file.")

    selected_file = data_files[0]
    print(f"Loading file from ZIP: {selected_file}")

    if selected_file.lower().endswith(".csv"):
        df = pd.read_csv(zip_file.open(selected_file))
    else:
        df = pd.read_excel(zip_file.open(selected_file))

print("Dataset loaded successfully.")
display(df.head())

## 2. Basic dataset understanding

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

## 3. Normal EDA

This section checks:
- Missing values
- Duplicate rows
- Numerical and categorical columns
- Statistical summary
- Categorical value counts

In [ ]:
# Missing values
missing_df = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
}).sort_values(by="Missing Count", ascending=False)

print("Missing values:")
display(missing_df)

# Duplicate rows
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

# Numerical and categorical columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("\nNumerical columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumerical summary:")
display(df[numeric_cols].describe())

if len(categorical_cols) > 0:
    print("\nCategorical summary:")
    display(df[categorical_cols].describe())

### Categorical value counts

In [ ]:
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print("Unique values:", df[col].nunique())
    display(df[col].value_counts().head(10))

## 4. EDA visualizations

In [ ]:
# Histograms for numerical columns
for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# Boxplots for outlier checking
for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.show()

In [ ]:
# Correlation heatmap
if len(numeric_cols) > 1:
    plt.figure(figsize=(12, 8))
    corr_matrix = df[numeric_cols].corr()
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap")
    plt.show()
else:
    print("Not enough numerical columns for correlation heatmap.")

In [ ]:
# Bar plots for categorical columns
for col in categorical_cols:
    plt.figure(figsize=(8, 4))
    df[col].value_counts().head(10).plot(kind="bar")
    plt.title(f"Top categories in {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()

## 5. Handle missing values and duplicates

For this assessment:
- Duplicate rows are removed.
- Numerical missing values are filled with median.
- Categorical missing values are filled with mode.

In [ ]:
df_clean = df.copy()

# Remove duplicates
before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
after = df_clean.shape[0]
print(f"Removed duplicate rows: {before - after}")

# Fill missing values
numeric_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    if df_clean[col].mode().empty:
        df_clean[col] = df_clean[col].fillna("Unknown")
    else:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Total missing values after cleaning:", df_clean.isnull().sum().sum())
display(df_clean.head())

## 6. Outlier detection and treatment using IQR capping

Outliers are capped instead of removed so that we do not lose important records.

In [ ]:
def cap_outliers_iqr(data, columns):
    data = data.copy()
    outlier_report = []

    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

        outlier_count = ((data[col] < lower_limit) | (data[col] > upper_limit)).sum()

        data[col] = np.where(data[col] < lower_limit, lower_limit, data[col])
        data[col] = np.where(data[col] > upper_limit, upper_limit, data[col])

        outlier_report.append({
            "Column": col,
            "Lower Limit": lower_limit,
            "Upper Limit": upper_limit,
            "Outlier Count": outlier_count
        })

    return data, pd.DataFrame(outlier_report)

numeric_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
df_clean, outlier_report = cap_outliers_iqr(df_clean, numeric_cols)

print("Outlier report:")
display(outlier_report)

## 7. Feature engineering

Examples included:
- Dosage-to-volume ratio, if dosage and volume columns are available.
- Temperature bands, if a temperature column is available.

In [ ]:
df_featured = df_clean.copy()

# Helper function to find column by keyword
def find_column(columns, keywords):
    for col in columns:
        col_lower = col.lower().replace(" ", "_")
        if any(keyword in col_lower for keyword in keywords):
            return col
    return None

all_cols = df_featured.columns.tolist()

dosage_col = find_column(all_cols, ["dosage", "dose"])
volume_col = find_column(all_cols, ["volume", "vol"])
temperature_col = find_column(all_cols, ["temperature", "temp"])

created_features = []

# Dosage to volume ratio
if dosage_col is not None and volume_col is not None:
    df_featured["dosage_to_volume_ratio"] = df_featured[dosage_col] / (df_featured[volume_col] + 1e-6)
    created_features.append("dosage_to_volume_ratio")

# Temperature band
if temperature_col is not None:
    df_featured["temperature_band"] = pd.cut(
        df_featured[temperature_col],
        bins=[-np.inf, 15, 25, 35, np.inf],
        labels=["Cold", "Normal", "Warm", "Hot"]
    )
    created_features.append("temperature_band")

print("Created features:", created_features if created_features else "No automatic feature was created because matching columns were not found.")
display(df_featured.head())

## 8. Target variable selection

For shelf-life prediction, the target should ideally be a shelf-life, expiry, or stability column.

If the automatic target selection is wrong, manually change `TARGET_COLUMN`.

In [ ]:
# You can manually set target column here if needed.
# Example:
# TARGET_COLUMN = "Shelf_Life_Days"

TARGET_COLUMN = None

preferred_targets = [
    "Shelf_Life_Days",
    "Shelf_Life",
    "shelf_life",
    "Expiry_Days",
    "Expiration_Days",
    "Stability_Days",
    "Demand_Forecast"   # fallback because your earlier notebook used this column
]

if TARGET_COLUMN is None:
    for target in preferred_targets:
        if target in df_featured.columns:
            TARGET_COLUMN = target
            break

if TARGET_COLUMN is None:
    target_keywords = ["shelf", "life", "expiry", "expiration", "stability", "forecast", "demand", "target"]
    possible_targets = [
        col for col in df_featured.columns
        if any(keyword in col.lower() for keyword in target_keywords)
    ]

    print("Possible target columns:", possible_targets)

    if len(possible_targets) > 0:
        TARGET_COLUMN = possible_targets[0]
    else:
        # Last fallback: use the last numerical column
        numeric_cols_temp = df_featured.select_dtypes(include=["int64", "float64"]).columns.tolist()
        TARGET_COLUMN = numeric_cols_temp[-1]

print("Selected target column:", TARGET_COLUMN)

# Make target numeric
df_featured[TARGET_COLUMN] = pd.to_numeric(df_featured[TARGET_COLUMN], errors="coerce")
df_featured = df_featured.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

print("Target summary:")
display(df_featured[TARGET_COLUMN].describe())

plt.figure(figsize=(7, 4))
sns.histplot(df_featured[TARGET_COLUMN], kde=True)
plt.title(f"Target Variable Distribution: {TARGET_COLUMN}")
plt.xlabel(TARGET_COLUMN)
plt.ylabel("Frequency")
plt.show()

## 9. Label Encoding

This section converts categorical values into numeric values using **LabelEncoder**.

In [ ]:
df_encoded = df_featured.copy()

categorical_cols = df_encoded.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

print("Label Encoding completed.")
print("Encoded categorical columns:", categorical_cols)

display(df_encoded.head())

In [ ]:
# Check label mapping for each categorical column
for col, le in label_encoders.items():
    mapping = pd.DataFrame({
        "Original Category": le.classes_,
        "Encoded Value": range(len(le.classes_))
    })

    print(f"\nMapping for column: {col}")
    display(mapping.head(20))

## 10. Split data into train and test sets

In [ ]:
X = df_encoded.drop(columns=[TARGET_COLUMN])
y = df_encoded[TARGET_COLUMN]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 11. Model building and evaluation functions

In [ ]:
def evaluate_regression_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return {
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "R2 Score": r2
    }, y_pred

## 12. Linear Regression

In [ ]:
linear_model = LinearRegression()

linear_result, linear_pred = evaluate_regression_model(
    linear_model,
    X_train,
    X_test,
    y_train,
    y_test,
    "Linear Regression"
)

linear_result

## 13. Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE
)

rf_result, rf_pred = evaluate_regression_model(
    rf_model,
    X_train,
    X_test,
    y_train,
    y_test,
    "Random Forest Regressor"
)

rf_result

## 14. Gradient Boosting Regressor

In [ ]:
gb_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gb_result, gb_pred = evaluate_regression_model(
    gb_model,
    X_train,
    X_test,
    y_train,
    y_test,
    "Gradient Boosting Regressor"
)

gb_result

## 15. Model comparison

In [ ]:
results_df = pd.DataFrame([linear_result, rf_result, gb_result])
results_df = results_df.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

display(results_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=results_df, x="Model", y="RMSE")
plt.title("Model Comparison Based on RMSE")
plt.xticks(rotation=30)
plt.show()

## 16. Hyperparameter tuning using GridSearchCV

Here, Random Forest is tuned using GridSearchCV.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

best_rf_model = grid_search.best_estimator_

## 17. Final model evaluation

In [ ]:
final_pred = best_rf_model.predict(X_test)

final_rmse = np.sqrt(mean_squared_error(y_test, final_pred))
final_mae = mean_absolute_error(y_test, final_pred)
final_r2 = r2_score(y_test, final_pred)

print("Final Tuned Random Forest Performance:")
print("RMSE:", final_rmse)
print("MAE:", final_mae)
print("R2 Score:", final_r2)

## 18. Cross-validation

In [ ]:
cv_mse_scores = cross_val_score(
    best_rf_model,
    X,
    y,
    cv=5,
    scoring="neg_mean_squared_error"
)

cv_rmse_scores = np.sqrt(-cv_mse_scores)

print("Cross-validation RMSE scores:", cv_rmse_scores)
print("Mean CV RMSE:", cv_rmse_scores.mean())
print("Standard Deviation CV RMSE:", cv_rmse_scores.std())

## 19. Actual vs predicted values

In [ ]:
prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": final_pred
})

display(prediction_df.head(20))

plt.figure(figsize=(7, 5))
plt.scatter(y_test, final_pred)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted Values")
plt.show()

## 20. Feature importance

In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

display(feature_importance_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=feature_importance_df.head(15), x="Importance", y="Feature")
plt.title("Top 15 Feature Importances")
plt.show()

## 21. Save final model

In [ ]:
model_package = {
    "model": best_rf_model,
    "label_encoders": label_encoders,
    "target_column": TARGET_COLUMN,
    "feature_columns": X.columns.tolist()
}

joblib.dump(model_package, "drug_shelf_life_final_model.pkl")

print("Model saved as drug_shelf_life_final_model.pkl")

## 22. Final conclusion

The dataset was cleaned by handling missing values, removing duplicates, and capping outliers. Normal EDA was performed using summaries, histograms, boxplots, categorical value counts, and correlation analysis. Categorical columns were encoded using LabelEncoder. Three regression models were trained: Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor. The models were compared using RMSE, MAE, and R² score. Random Forest was tuned using GridSearchCV, and the final tuned model was evaluated using test-set metrics and cross-validation RMSE.